# Stock Return Forecasting and Portfolio Construction Using an Event-Conditional Mixture-of-Experts Framework

## 1. Libraries

In [1]:
import os
import setuptools
os.environ["OMP_NUM_THREADS"] = "6"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import re
import datetime
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
import tensorflow as tf
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (Dense, Dropout, SpatialDropout1D, LSTM, GRU, LayerNormalization,
                                     MultiHeadAttention, GlobalAveragePooling1D, Concatenate, Lambda)
from statsmodels.tsa.arima.model import ARIMA
from arch import arch_model
from scipy import stats
from IPython.display import display


## 2. Import data

In [2]:
DATA_DIR   = "Initial stock data"
PREDS_DIR  = "Final predictions"
FIG_DIR    = "figures"

RETRAIN = False
DOWNLOAD_PRICES = False

CFG = {
    "TICKERS": ["NVDA","AAPL","TSM","JPM","ASML","XOM","TCEHY","MA","BAC","GE",
                "CVX","PG","MS","KO","HD","HSBC","GS","NVS","PM","RY",
                "AZN","BABA","WFC","IBM","C","BHP","MUFG","SHEL","TM","TD",
                "VZ","SAP","NVO","SAN","AMGN","TTE","NEE","TMO","RIO","BLK"],
    "START": "2000-01-01", "END": "2026-05-31",
    "LOOKBACK": 30, "TARGET": "y_ret",
    "TRAIN_END": "2021-01-01", "VAL_END": "2023-01-01",
    "EPOCHS": 500, "SEED": 42,
    "COST_BPS": 5.0, "MIN_WEIGHT": 0.01,
    "CAPACITY": {"LSTM": {"units": [256,128,64,32]},
                 "GRU": {"units": [128,64,32]},
                 "TRANSFORMER": {"units": [128,64,32], "heads": 4}},
    "PERIODS": {"2000s":  ["2000-01-01","2010-12-31"],
                "2010s":  ["2011-01-01","2020-12-31"],
                "whole":  ["2000-01-01","2022-01-30"],
                "middle": ["2013-01-01","2022-12-31"]},
}
HORIZONS = {
    "1D":  {"kind": "days",      "n": 1,      "label": "next-day return"},
    "1W":  {"kind": "days",      "n": 5,      "label": "1-week return (5 trading days)"},
    "1ME": {"kind": "month_end", "months": 1, "label": "to last business day of next month"},
}
TICKERS       = CFG["TICKERS"]
FREQ_DAYS     = {"1D": 1, "1W": 5, "1ME": 21}
DM_CANDIDATES = ["ARIMA", "single_LSTM", "Event_MoE", "Event_MoE_Regime", "Regime_Pooled", "Regime_Pooled_Aug"]
METHODS       = ["TopK", "ScoreWeighted", "MaxSharpe", "MinVariance", "TopKInvVol", "EqualWeight_BuyHold"]
print("tickers", len(TICKERS), "| L", CFG["LOOKBACK"],
      "| EPOCHS", CFG["EPOCHS"], "| seed", CFG["SEED"], "| cost_bps", CFG["COST_BPS"])


tickers 40 | L 30 | EPOCHS 500 | seed 42 | cost_bps 5.0


In [3]:
PRICES = {}

def fetch_price(ticker):
    df = yf.download(ticker, start=CFG["START"], end=CFG["END"], auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [c[0] for c in df.columns]
    df.index = pd.to_datetime(df.index).tz_localize(None).normalize()
    return df[["Open", "High", "Low", "Close", "Volume"]].dropna()

def read_price_excel(ticker):
    df = pd.read_excel(os.path.join(DATA_DIR, f"df_{ticker}.xlsx"))
    df["Date"] = pd.to_datetime(df["Date"])
    return df.set_index("Date")[["Open", "High", "Low", "Close", "Volume"]].dropna()

def load_price(ticker, refresh=False):
    if ticker not in PRICES or refresh:
        PRICES[ticker] = fetch_price(ticker) if DOWNLOAD_PRICES else read_price_excel(ticker)
    return PRICES[ticker]

def import_prices(tickers):
    for t in tickers:
        df = load_price(t)
        if DOWNLOAD_PRICES:
            df.reset_index().rename(columns={"index": "Date"}).to_excel(
                os.path.join(DATA_DIR, f"df_{t}.xlsx"), index=False)
        print(f"  {t:6s} rows={len(df):5d}  {df.index.min().date()} -> {df.index.max().date()}")
    print(f"{len(tickers)} tickers " + ("downloaded from Yahoo and saved" if DOWNLOAD_PRICES
          else f"read from {DATA_DIR}/"))

import_prices(TICKERS)


  NVDA   rows= 6641  2000-01-03 -> 2026-05-29
  AAPL   rows= 6641  2000-01-03 -> 2026-05-29
  TSM    rows= 6641  2000-01-03 -> 2026-05-29
  JPM    rows= 6641  2000-01-03 -> 2026-05-29
  ASML   rows= 6641  2000-01-03 -> 2026-05-29
  XOM    rows= 6641  2000-01-03 -> 2026-05-29
  TCEHY  rows= 4125  2010-01-05 -> 2026-05-29
  MA     rows= 5034  2006-05-25 -> 2026-05-29
  BAC    rows= 6641  2000-01-03 -> 2026-05-29
  GE     rows= 6641  2000-01-03 -> 2026-05-29
  CVX    rows= 6641  2000-01-03 -> 2026-05-29
  PG     rows= 6641  2000-01-03 -> 2026-05-29
  MS     rows= 6641  2000-01-03 -> 2026-05-29
  KO     rows= 6641  2000-01-03 -> 2026-05-29
  HD     rows= 6641  2000-01-03 -> 2026-05-29
  HSBC   rows= 6641  2000-01-03 -> 2026-05-29
  GS     rows= 6641  2000-01-03 -> 2026-05-29
  NVS    rows= 6641  2000-01-03 -> 2026-05-29
  PM     rows= 4580  2008-03-17 -> 2026-05-29
  RY     rows= 6641  2000-01-03 -> 2026-05-29
  AZN    rows= 6641  2000-01-03 -> 2026-05-29
  BABA   rows= 2940  2014-09-19 ->

In [4]:
def load_events():
    df = pd.read_csv("macro_events_combined.csv", parse_dates=["date"])
    return df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

print("events:", len(load_events()), "rows")


events: 869 rows


## 3. Other functions

In [5]:
PRED_STORE = {}
BOOK_STORE = {}
VOL_STORE = {}

def load_predictions(ticker, model, horizon=None, target=None) -> pd.DataFrame:
    target = target or CFG["TARGET"]
    key = (ticker, model, horizon, target)
    if key not in PRED_STORE:
        raise FileNotFoundError(f"no predictions for {ticker} · {model} · {horizon} · {target} "
                                f"— run the training section first")
    return PRED_STORE[key].copy()

def vol_model_features(ticker) -> pd.DataFrame:
    if ticker not in VOL_STORE:
        VOL_STORE[ticker] = compute_vol_models(load_price(ticker))
    return VOL_STORE[ticker]

def load_saved_predictions():
    n = 0
    for fn in sorted(os.listdir(PREDS_DIR)):
        if not fn.endswith("_preds.xlsx"):
            continue
        t = fn[:-len("_preds.xlsx")]
        xl = pd.ExcelFile(os.path.join(PREDS_DIR, fn))
        for h in xl.sheet_names:
            d = xl.parse(h)
            if "date" not in d.columns:
                continue
            for col in [c for c in d.columns if c.startswith("y_pred_")]:
                m = col[len("y_pred_"):]
                PRED_STORE[(t, m, h, CFG["TARGET"])] = d[["date", "y_true", col]].rename(columns={col: "y_pred"})
        n += 1
    print(f"loaded {len(PRED_STORE)} prediction series from {PREDS_DIR}/ ({n} tickers)")


In [6]:
def technical(df):
    c, h, l = df["Close"], df["High"], df["Low"]
    f = pd.DataFrame(index=df.index)
    f["logret1"]  = np.log(c).diff()
    f["vol20"]    = f["logret1"].rolling(20).std()
    f["logret20"] = np.log(c).diff(20)
    f["range20"]  = (c.rolling(20).max() - c.rolling(20).min()) / c
    f["dd60"]     = c / c.rolling(60).max() - 1
    f["hl_range"] = (h - l) / c
    return f

def macro_features(index, clip=5):
    ev = load_events(); ti = pd.DatetimeIndex(index)
    out = pd.DataFrame(index=index)
    for et in ["FOMC", "CPI", "NFP"]:
        de = pd.DatetimeIndex(sorted(ev.loc[ev["event_type"] == et, "date"].unique()))
        if len(de) == 0:
            out[f"days_to_{et}"] = clip; out[f"is_{et}"] = 0; continue
        pos = de.searchsorted(ti, side="left")
        nxt = np.where(pos < len(de), de[np.clip(pos, 0, len(de) - 1)].values, np.datetime64("NaT"))
        d2 = (pd.DatetimeIndex(nxt) - ti).days.astype("float")
        d2 = np.where(np.isnan(d2), clip, np.clip(d2, 0, clip))
        out[f"days_to_{et}"] = d2.astype(int); out[f"is_{et}"] = (out[f"days_to_{et}"] == 0).astype(int)
    return out

def compute_vol_models(px):
    r = (np.log(px["Close"].astype(float)).diff().dropna() * 100.0)
    rv = r.values ** 2
    n = len(r)
    rv_s = pd.Series(rv, index=r.index)
    har_X = np.column_stack([np.ones(n), rv,
                             rv_s.rolling(5).mean().values,
                             rv_s.rolling(22).mean().values])
    garch = np.full(n, np.nan); har = np.full(n, np.nan)
    mu = omega = alpha = beta = state = har_b = None
    for t in range(VOL_WARMUP, n):
        if (t - VOL_WARMUP) % VOL_REFIT_DAYS == 0:
            try:
                res = arch_model(r.values[:t + 1], mean="Constant", vol="GARCH",
                                 p=1, q=1, rescale=False).fit(disp="off", show_warning=False)
                pm = res.params
                mu, omega = float(pm["mu"]), float(pm["omega"])
                alpha, beta = float(pm["alpha[1]"]), float(pm["beta[1]"])
                state = float(res.conditional_volatility[-1]) ** 2
            except Exception as e:
                print(f"GARCH fit failed at {r.index[t].date()}: {e}")
            valid = np.where(~np.isnan(har_X).any(axis=1))[0]
            s_rows = valid[valid <= t - 1]
            if len(s_rows) > 50:
                har_b, *_ = np.linalg.lstsq(har_X[s_rows], rv[s_rows + 1], rcond=None)
        if omega is not None:
            state = omega + alpha * (r.values[t] - mu) ** 2 + beta * state
            garch[t] = state
        if har_b is not None and not np.isnan(har_X[t]).any():
            har[t] = max(float(har_X[t] @ har_b), 0.0)
    return pd.DataFrame({"garch_var": garch, "har_rv": har}, index=r.index).dropna()

def build_features(ticker, horizon=None, vol_models=False):
    spec = HORIZONS[horizon]
    px = load_price(ticker)
    f = technical(px)
    if vol_models:
        f = f.join(vol_model_features(ticker))
    c = px["Close"].astype(float)
    lc = np.log(c)

    if spec["kind"] == "days":
        n = int(spec["n"])
        fwd = (lc.shift(-n) - lc).reindex(f.index).values
        tdate = pd.Series(c.index, index=c.index).shift(-n).reindex(f.index)
        f["days_to_target"] = (pd.to_datetime(tdate) - pd.Series(f.index, index=f.index)).dt.days
    else:
        months = int(spec["months"])
        tgt = pd.DatetimeIndex([((pd.Timestamp(d) + pd.DateOffset(months=months)) + pd.offsets.BMonthEnd(0)) for d in f.index])
        pos = c.index.searchsorted(tgt, side="right") - 1
        fwd_close = np.where(tgt > c.index[-1], np.nan, c.values[np.clip(pos, 0, len(c) - 1)])
        fwd = np.log(fwd_close) - lc.reindex(f.index).values
        f["days_to_target"] = (tgt - f.index).days

    f = f.join(macro_features(f.index))
    f["y_ret"] = fwd
    return f.dropna()

def make_windows(f, target, lookback):
    feats = f.drop(columns=[c for c in ["y_ret"] if c in f.columns])
    X = feats.values.astype("float32"); y = f[target].values.astype("float32")
    Xw = np.stack([X[i - lookback:i] for i in range(lookback, len(X))])
    return Xw, y[lookback:], f.index[lookback:], feats.columns.tolist()

def standardize_windows(Xw, tr):
    Xtr = Xw[tr]
    mu = Xtr.mean(axis=(0, 1), keepdims=True)
    sd = Xtr.std(axis=(0, 1), keepdims=True) + 1e-8
    return ((Xw - mu) / sd).astype("float32")

def capacity(model):
    d = CFG["CAPACITY"][model]
    return [int(x) for x in d["units"] if int(x) > 0], max(1, int(d.get("heads", 4)))

def phi(Xwin):
    return np.concatenate([Xwin.mean(axis=0), Xwin.std(axis=0)])


In [7]:
def portfolio_metrics(nr: pd.Series, bond: float = 0.02) -> dict:
    r = nr.dropna()
    if len(r) < 2:
        return dict(ann_ret=float("nan"), ann_vol=float("nan"), Sharpe=float("nan"),
                    Sortino=float("nan"), Calmar=float("nan"), max_dd=float("nan"),
                    hit=float("nan"), n=int(len(r)))
    ann_ret = float((1 + r).prod() ** (252 / len(r)) - 1)
    ann_vol = float(r.std(ddof=1) * np.sqrt(252))
    nav = (1 + r).cumprod(); dd = float((nav / nav.cummax() - 1).min())
    dn = r[r < 0].std(ddof=1) * np.sqrt(252)
    return dict(ann_ret=ann_ret, ann_vol=ann_vol,
                Sharpe=(ann_ret - bond) / ann_vol if ann_vol > 0 else float("nan"),
                Sortino=(ann_ret - bond) / dn if dn and dn > 0 else float("nan"),
                Calmar=ann_ret / abs(dd) if dd < 0 else float("nan"),
                max_dd=dd, hit=float((r > 0).mean()), n=int(len(r)))

def score_price_panels(model, horizon, tickers):
    scores = {}
    for t in tickers:
        df = load_predictions(t, model, horizon).dropna(subset=["y_pred"]).copy()
        df["date"] = pd.to_datetime(df["date"])
        scores[t] = df.set_index("date")["y_pred"]
    S = pd.DataFrame(scores).sort_index().dropna(how="all")
    raw = pd.DataFrame({t: load_price(t)["Close"] for t in tickers}).sort_index()
    if len(S):
        idx = raw.index[raw.index >= S.index[0]]
        P = raw.reindex(idx).ffill()
    else:
        P = raw.reindex(S.index).ffill()
    return S, P

def min_weight() -> float:
    try:
        return max(0.0, float(CFG.get("MIN_WEIGHT", 0.01)))
    except (TypeError, ValueError):
        return 0.01

def apply_floor(wdict: dict, min_w: float) -> dict:
    if min_w <= 0 or not wdict:
        return wdict
    kept = {t: w for t, w in wdict.items() if w >= min_w}
    s = sum(kept.values())
    if not kept or s <= 0:
        return wdict
    return {t: w / s for t, w in kept.items()}

def method_weights(method, dt, S, R, cols, K, lookback=60):
    n = len(cols)
    if method == "TopK":
        row = S.loc[dt]
        z = (row - row.mean()) / (row.std() + 1e-9)
        picks = list(z.dropna().sort_values(ascending=False).index[:K])
        return {t: 1.0 / len(picks) for t in picks} if picks else {}
    if method == "ScoreWeighted":
        row = S.loc[dt]
        z = ((row - row.mean()) / (row.std() + 1e-9)).reindex(cols).fillna(0.0)
        pos = z.clip(lower=0.0)
        s = float(pos.sum())
        return apply_floor((pos / s).to_dict(), min_weight()) if s > 0 else {t: 1.0 / n for t in cols}
    if method == "EqualWeight_BuyHold":
        return {t: 1.0 / n for t in cols}
    win = R.loc[:dt].tail(lookback)[cols]
    if len(win) < 5:
        return {t: 1.0 / n for t in cols}
    if method == "TopKInvVol":
        row = S.loc[dt]
        z = (row - row.mean()) / (row.std() + 1e-9)
        picks = list(z.dropna().sort_values(ascending=False).index[:K])
        if not picks:
            return {}
        inv = (1.0 / win[picks].std().replace(0, np.nan)).reindex(picks).fillna(0.0)
        s = float(inv.sum())
        return (inv / s).to_dict() if s > 0 else {t: 1.0 / len(picks) for t in picks}
    Sig = np.nan_to_num(win.cov().values)
    Sig = Sig + np.eye(n) * (np.trace(Sig) / n * 1e-3 + 1e-8)
    try:
        if method == "MinVariance":
            w = np.linalg.solve(Sig, np.ones(n))
        else:
            mu = np.nan_to_num(win.mean().reindex(cols).values)
            w = np.linalg.solve(Sig, mu)
    except np.linalg.LinAlgError:
        w = np.ones(n)
    w = np.clip(w, 0, None)
    if not np.isfinite(w).all() or w.sum() <= 0:
        w = np.ones(n)
    w = w / w.sum()
    return apply_floor(dict(zip(cols, w)), min_weight())

def rebal_dates(method, idx, freq):
    return [idx[0]] if (method == "EqualWeight_BuyHold" and idx) else idx[::max(freq, 1)]

def backtest(weight_fn, P, cost_bps, bond, rebal):
    rets = P.pct_change().fillna(0.0)
    bond_daily = (1 + bond) ** (1 / 252) - 1
    cols = list(P.columns); rebal = set(rebal)
    w = pd.Series(0.0, index=cols)
    out = []
    for dt in list(P.index):
        r = rets.loc[dt]
        cash = 1.0 - float(w.sum())
        r_p = float((w * r).sum()) + cash * bond_daily
        g = 1.0 + r_p
        if g > 0:
            w = (w * (1 + r)) / g
        cost = 0.0
        if dt in rebal:
            tgt = pd.Series(0.0, index=cols)
            for t, v in weight_fn(dt).items():
                if t in tgt.index:
                    tgt[t] = v
            cost = cost_bps / 1e4 * float((tgt - w).abs().sum())
            w = tgt
        out.append((dt, r_p - cost))
    return pd.Series(dict(out), name="net_ret")

def build_portfolio(model, horizon, tickers, K=None, cost_bps=None, bond=0.02, capital=10000.0,
                    method="TopK"):
    cost_bps = float(CFG.get("COST_BPS", 1.0)) if cost_bps is None else cost_bps
    freq = FREQ_DAYS.get(horizon, 21)
    K = K or max(1, len(tickers) // 2)
    S, P = score_price_panels(model, horizon, tickers)
    R = P.pct_change().fillna(0.0)
    cols = list(P.columns); idx = list(S.index)
    nr = backtest(lambda dt: method_weights(method, dt, S, R, cols, K),
                   P, cost_bps, bond, rebal_dates(method, idx, freq))
    bench = backtest(lambda dt: {t: 1.0 / len(cols) for t in cols},
                      P, cost_bps, bond, idx[::max(freq, 1)])
    cfg = dict(model=model, horizon=horizon, tickers=list(tickers), K=K, method=method,
               freq_days=freq, cost_bps=cost_bps, bond=bond, capital=float(capital),
               min_weight=min_weight())
    return nr, bench, portfolio_metrics(nr, bond), cfg

def trade_ledger(model, horizon, tickers, K=None, cost_bps=None, bond=0.02, capital=10000.0,
                 method="TopK") -> dict:
    cost_bps = float(CFG.get("COST_BPS", 1.0)) if cost_bps is None else cost_bps
    freq = FREQ_DAYS.get(horizon, 21)
    K = K or max(1, len(tickers) // 2)
    S, P = score_price_panels(model, horizon, tickers)
    R = P.pct_change().fillna(0.0)
    cols = list(P.columns)
    rebal = set(rebal_dates(method, list(S.index), freq))
    bond_daily = (1 + bond) ** (1 / 252) - 1
    fee = cost_bps / 1e4

    shares = {t: 0.0 for t in cols}
    avg_cost = {t: 0.0 for t in cols}
    cash = float(capital)
    realized_cum = fees_cum = 0.0
    trades, daily = [], []

    for dt in list(P.index):
        price = P.loc[dt]
        if dt in rebal:
            equity = cash + sum(shares[t] * float(price[t]) for t in cols if np.isfinite(price[t]))
            tgt_w = method_weights(method, dt, S, R, cols, K)
            for t in cols:
                px = float(price[t])
                if not np.isfinite(px) or px <= 0:
                    continue
                trade_val = tgt_w.get(t, 0.0) * equity - shares[t] * px
                if abs(trade_val) < equity * 1e-6:
                    continue
                d_sh = trade_val / px
                cost = abs(trade_val) * fee
                fees_cum += cost
                if d_sh > 0:
                    new_sh = shares[t] + d_sh
                    avg_cost[t] = (avg_cost[t] * shares[t] + px * d_sh) / new_sh
                    shares[t] = new_sh
                    cash -= trade_val + cost
                    trades.append(dict(date=dt, ticker=t, side="BUY", shares=d_sh, price=px,
                                       value=trade_val, fee=cost, realized=0.0))
                else:
                    sell_sh = -d_sh
                    realized = (px - avg_cost[t]) * sell_sh
                    realized_cum += realized
                    shares[t] += d_sh
                    cash += sell_sh * px - cost
                    trades.append(dict(date=dt, ticker=t, side="SELL", shares=sell_sh, price=px,
                                       value=sell_sh * px, fee=cost, realized=realized))
        cash *= (1 + bond_daily)
        pos_val = sum(shares[t] * float(price[t]) for t in cols if np.isfinite(price[t]))
        unreal = sum((float(price[t]) - avg_cost[t]) * shares[t] for t in cols
                     if shares[t] > 1e-12 and np.isfinite(price[t]))
        daily.append(dict(date=dt, cash=cash, positions=pos_val, equity=cash + pos_val,
                          realized=realized_cum, unrealized=unreal, fees=fees_cum))

    trades_df = pd.DataFrame(trades)
    daily_df = pd.DataFrame(daily).set_index("date") if daily else pd.DataFrame()
    last = P.iloc[-1]
    pos = [dict(ticker=t, shares=shares[t], avg_cost=avg_cost[t], last=float(last[t]),
                market_value=shares[t] * float(last[t]), unrealized=(float(last[t]) - avg_cost[t]) * shares[t])
           for t in cols if shares[t] > 1e-9 and np.isfinite(last[t])]
    pos_df = pd.DataFrame(pos)
    final_eq = float(daily_df["equity"].iloc[-1]) if len(daily_df) else float(capital)
    summary = dict(capital=float(capital), final_equity=final_eq, net_pnl=final_eq - float(capital),
                   total_return=(final_eq / float(capital) - 1) if capital else float("nan"),
                   realized=realized_cum,
                   unrealized=float(daily_df["unrealized"].iloc[-1]) if len(daily_df) else 0.0,
                   fees=fees_cum, n_trades=len(trades_df), cost_bps=cost_bps)
    return dict(trades=trades_df, daily=daily_df, positions=pos_df, summary=summary)


In [8]:
def build_and_save(model, horizon, tickers, K=None, cost_bps=None, bond=0.02, capital=10000.0,
                   method="TopK", force=False):
    cost_bps = float(CFG.get("COST_BPS", 1.0)) if cost_bps is None else cost_bps
    K_eff = min(max(1, K), len(tickers)) if K else max(1, len(tickers) // 2)
    uses_k = method in ("TopK", "TopKInvVol")
    name = f"{model}-{horizon}-{method}" + (f"-K{K_eff}" if uses_k else "")
    if name in BOOK_STORE and not force:
        BOOK_STORE[name]["capital"] = float(capital)
        return name, True
    nr, bench, mtr, cfg = build_portfolio(model, horizon, tickers, K_eff, cost_bps, bond, capital, method)
    returns = pd.DataFrame({"net_ret": nr.values,
                            "bench_1N_rebalanced": bench.reindex(nr.index).values}, index=nr.index)
    returns.index.name = "date"
    BOOK_STORE[name] = {"name": name, **cfg, "metrics": mtr, "returns": returns}
    return name, False

def sweep_real(target_metric: str = "Sharpe") -> pd.DataFrame:
    rows = []
    for name, meta in BOOK_STORE.items():
        m = meta.get("metrics", {}) or {}
        method = meta.get("method", "TopK"); kval = meta.get("K")
        rows.append(dict(portfolio=name, model=meta.get("model"), horizon=meta.get("horizon"),
                         method=method,
                         K=(str(int(kval)) if method in ("TopK", "TopKInvVol") and kval is not None else ""),
                         Sharpe=m.get("Sharpe"), ann_ret=m.get("ann_ret"), ann_vol=m.get("ann_vol"),
                         Sortino=m.get("Sortino"), Calmar=m.get("Calmar"), max_dd=m.get("max_dd"),
                         hit=m.get("hit"), n=m.get("n")))
    cols = ["portfolio","model","horizon","method","K","Sharpe","ann_ret","ann_vol",
            "Sortino","Calmar","max_dd","hit","n"]
    if not rows:
        return pd.DataFrame(columns=cols)
    return pd.DataFrame(rows)[cols].sort_values(target_metric, ascending=False,
                                                na_position="last").reset_index(drop=True)

def sharpe_test_portfolio_real(horizon=None, method="TopK", K=7) -> pd.DataFrame:
    uses_k = method in ("TopK", "TopKInvVol"); ann = np.sqrt(252); rows = []
    for m in ["NAIVE"] + DM_CANDIDATES:
        name = f"{m}-{horizon}-{method}" + (f"-K{K}" if uses_k else "")
        if name not in BOOK_STORE:
            continue
        df = BOOK_STORE[name]["returns"]
        if "bench_1N_rebalanced" not in df.columns:
            continue
        df = df[["net_ret", "bench_1N_rebalanced"]].dropna()
        rp, rb = df["net_ret"].values, df["bench_1N_rebalanced"].values
        T = len(rp)
        if T < 30:
            continue
        sp, sb = rp.std(ddof=1), rb.std(ddof=1)
        if sp <= 0 or sb <= 0:
            continue
        SRp, SRb = rp.mean() / sp, rb.mean() / sb
        rho = float(np.corrcoef(rp, rb)[0, 1])
        theta = (1.0 / T) * (2 * (1 - rho) + 0.5 * (SRp**2 + SRb**2 - 2 * SRp * SRb * rho**2))
        z = (SRp - SRb) / np.sqrt(max(theta, 1e-18)); pv = float(2 * stats.norm.sf(abs(z)))
        rows.append(dict(model=m, n=T, sharpe=round(SRp * ann, 3), sharpe_1N=round(SRb * ann, 3),
                         z=round(z, 3), p_value=round(pv, 4),
                         verdict=("higher ✓" if z > 0 and pv < 0.05 else
                                  "lower ✗" if z < 0 and pv < 0.05 else "n.s.")))
    return pd.DataFrame(rows, columns=["model","n","sharpe","sharpe_1N","z","p_value","verdict"])


In [9]:
def dm_pred(d, h):
    n = len(d); dbar = float(d.mean()); dd = d - dbar; lag = max(0, h - 1)
    lrv = float(np.mean(dd * dd))
    for k in range(1, lag + 1):
        lrv += 2.0 * (1 - k / (lag + 1)) * float(np.mean(dd[k:] * dd[:-k]))
    lrv = max(lrv, 1e-18)
    dm = dbar / np.sqrt(lrv / n)
    hln = np.sqrt(max((n + 1 - 2 * h + h * (h - 1) / n) / n, 1e-9)) if n > h else 1.0
    dm *= hln
    return dm, float(2 * stats.t.sf(abs(dm), max(n - 1, 1)))

def model_pred_summary(model: str, horizon: str | None = None,
                       target: str | None = None) -> dict:
    ics, rankics, rmsen, das = [], [], [], []
    mses, rmses, maes, r2s, biases = [], [], [], [], []
    n_ok, missing = 0, []
    for t in CFG["TICKERS"]:
        try:
            df = load_predictions(t, model, horizon, target).dropna()
        except FileNotFoundError:
            missing.append(t); continue
        if not len(df):
            missing.append(t); continue
        y, p = df["y_true"].values.astype(float), df["y_pred"].values.astype(float)
        err = y - p
        if p.std() > 1e-12 and y.std() > 1e-12:
            ics.append(float(np.corrcoef(y, p)[0, 1]))
            rankics.append(float(pd.Series(y).corr(pd.Series(p), method="spearman")))
        das.append(float(np.mean(np.sign(y) == np.sign(p))))
        mse = float(np.mean(err ** 2)); rmse = float(np.sqrt(mse)); rn = float(np.sqrt(np.mean(y ** 2)))
        ss_tot = float(np.sum((y - y.mean()) ** 2))
        mses.append(mse); rmses.append(rmse); maes.append(float(np.mean(np.abs(err))))
        biases.append(float(np.mean(p - y)))
        if ss_tot > 0:
            r2s.append(1 - float(np.sum(err ** 2)) / ss_tot)
        if rn > 0:
            rmsen.append(rmse / rn)
        n_ok += 1
    mean = lambda xs: float(np.mean(xs)) if xs else float("nan")
    return dict(model=model, available=(n_ok > 0 and not missing), n_tickers=n_ok,
                missing=missing, IC=mean(ics), rank_IC=mean(rankics),
                dir_acc=mean(das), RMSE_vs_naive=mean(rmsen),
                MSE=mean(mses), RMSE=mean(rmses), MAE=mean(maes), R2=mean(r2s), bias=mean(biases))

def dm_predictions_real(horizon: str | None = None, target: str | None = None) -> pd.DataFrame:
    h = FREQ_DAYS.get(horizon, 21)
    rows = []
    for t in CFG["TICKERS"]:
        try:
            base = load_predictions(t, "NAIVE", horizon, target)[["date", "y_true", "y_pred"]]
        except FileNotFoundError:
            continue
        base = base.rename(columns={"y_pred": "p_naive"})
        for m in DM_CANDIDATES:
            try:
                cand = load_predictions(t, m, horizon, target)[["date", "y_pred"]]
            except FileNotFoundError:
                continue
            j = base.merge(cand, on="date", how="inner").dropna()
            if len(j) < 20:
                continue
            e_naive = (j["y_true"] - j["p_naive"]).values
            e_cand = (j["y_true"] - j["y_pred"]).values
            d = e_naive ** 2 - e_cand ** 2
            dm, p = dm_pred(d, h)
            rows.append(dict(ticker=t, model=m, n=len(j), DM=round(dm, 3), p_value=round(p, 4),
                             verdict=("better ✓" if dm > 0 and p < 0.05 else
                                      "worse ✗" if dm < 0 and p < 0.05 else "n.s.")))
    return pd.DataFrame(rows, columns=["ticker", "model", "n", "DM", "p_value", "verdict"])

def window_labels(dates, scheme="year"):
    return pd.to_datetime(pd.Series(dates)).dt.year.astype(str).values

def walkforward_stability(models, horizon: str | None = None, scheme: str = "year",
                          target: str | None = None):
    per = {}
    for m in models:
        ys, ps, ws = [], [], []
        for t in CFG["TICKERS"]:
            try:
                df = load_predictions(t, m, horizon, target).dropna()
            except FileNotFoundError:
                continue
            if df.empty:
                continue
            ys.append(df["y_true"].values); ps.append(df["y_pred"].values)
            ws.append(window_labels(df["date"], scheme))
        if not ys:
            continue
        y, p, w = np.concatenate(ys), np.concatenate(ps), np.concatenate(ws)
        ic = {}
        for win in pd.unique(w):
            msk = w == win
            if msk.sum() > 30 and p[msk].std() > 1e-12 and y[msk].std() > 1e-12:
                ic[win] = float(np.corrcoef(y[msk], p[msk])[0, 1])
        if ic:
            per[m] = ic
    wins = sorted({win for ic in per.values() for win in ic})
    pw = pd.DataFrame([{"window": win, **{m: round(per.get(m, {}).get(win, float("nan")), 4) for m in models}}
                       for win in wins], columns=["window"] + list(models))
    srows = []
    for m in models:
        v = np.array([per.get(m, {}).get(win, np.nan) for win in wins], float)
        v = v[~np.isnan(v)]
        if len(v) == 0:
            continue
        sd = v.std(ddof=1) if len(v) > 1 else 0.0
        t_stat = float(v.mean() / (sd / np.sqrt(len(v)))) if sd > 0 else float("nan")
        srows.append(dict(model=m, n_windows=len(v), mean_IC=round(float(v.mean()), 4),
                          std_IC=round(float(sd), 4), pct_positive=round(float((v > 0).mean()), 3),
                          t_stat=round(t_stat, 2), min_IC=round(float(v.min()), 4),
                          max_IC=round(float(v.max()), 4),
                          verdict=("stable ✓" if (v > 0).mean() >= 0.6 and v.mean() > 0 else
                                   "shaky" if v.mean() > 0 else "no edge ✗")))
    summary = pd.DataFrame(srows, columns=["model", "n_windows", "mean_IC", "std_IC", "pct_positive",
                                           "t_stat", "min_IC", "max_IC", "verdict"])
    return pw, summary


## 4. Models

In [10]:
DROPOUT, ALPHA_DIR, LR = 0.2, 0.3, 1e-3
EPOCHS, BATCH, EVENT_DIM = 500, 128, 6
GATE_BALANCE_ALPHA = 0.01
VOL_REFIT_DAYS, VOL_WARMUP = 21, 250
REGIME_POOLED, REGIME_POOLED_AUG = {}, {}
REGIME_POOL_CAP, REGIME_POOL_EPOCHS = 8000, 40

def patience():
    ep = int(CFG.get("EPOCHS", EPOCHS))
    return min(max(ep // 10, 10), 25)


In [11]:
def build_keras(kind, input_shape):

    lstm_u, _   = capacity("LSTM")
    gru_u, _    = capacity("GRU")
    tr_u, th    = capacity("TRANSFORMER")

    def lstm_expert(inp):
        x = inp
        n = len(lstm_u)
        for i, u in enumerate(lstm_u):
            seq = i < n - 1
            x = LSTM(u, return_sequences=seq)(x)
            if seq:
                x = SpatialDropout1D(DROPOUT)(x)
        x = LayerNormalization()(x); x = Dropout(DROPOUT)(x); return Dense(1)(x)

    def gru_expert(inp):
        x = inp
        n = len(gru_u)
        for i, u in enumerate(gru_u):
            seq = i < n - 1
            x = GRU(u, return_sequences=seq)(x)
            if seq:
                x = SpatialDropout1D(DROPOUT)(x)
        x = LayerNormalization()(x); x = Dropout(DROPOUT)(x); return Dense(1)(x)

    def transformer_expert(inp):
        x = Dense(tr_u[0])(inp); cur = tr_u[0]
        for u in tr_u:
            if u != cur:
                x = Dense(u)(x); cur = u
            attn = MultiHeadAttention(num_heads=th, key_dim=u)(x, x)
            x = LayerNormalization()(x + attn)
            ff = Dense(u * 2, activation="relu")(x); ff = Dense(u)(ff)
            x = LayerNormalization()(x + ff)
        x = GlobalAveragePooling1D()(x); x = Dropout(DROPOUT)(x); return Dense(1)(x)

    def linear_expert(inp):
        last = Lambda(lambda z: z[:, -1, :])(inp)
        return Dense(1)(last)

    def event_gate(inp, n_experts=3):
        last = Lambda(lambda z: z[:, -1, :])(inp)
        base = Lambda(lambda z: z[:, :-EVENT_DIM])(last)
        events = Lambda(lambda z: z[:, -EVENT_DIM:])(last)
        b = Dense(32, activation="gelu")(base); e = Dense(16, activation="gelu")(events)
        x = Concatenate()([b, e]); x = Dropout(DROPOUT)(x)
        return Dense(n_experts, activation="softmax", name="gate")(x)

    inp = Input(shape=input_shape)
    if kind == "LSTM":
        out = lstm_expert(inp)
    elif kind == "MoE4":
        preds = Concatenate(axis=-1)([lstm_expert(inp), gru_expert(inp),
                                      transformer_expert(inp), linear_expert(inp)])
        w = event_gate(inp, 4)
        yhat = Lambda(lambda a: tf.reduce_sum(a[0] * a[1], axis=-1, keepdims=True))([w, preds])
        out = Concatenate(axis=-1)([yhat, w])
    else:
        preds = Concatenate(axis=-1)([lstm_expert(inp), gru_expert(inp), transformer_expert(inp)])
        w = event_gate(inp)
        out = Lambda(lambda a: tf.reduce_sum(a[0] * a[1], axis=-1, keepdims=True))([w, preds])
    model = Model(inp, out)

    huber = tf.keras.losses.Huber(delta=1.0)
    if kind == "MoE4":
        def loss(yt, yp):
            yt = tf.reshape(tf.cast(yt, yp.dtype), (-1, 1))
            yh, g = yp[:, :1], yp[:, 1:]
            s2 = tf.math.reduce_variance(yt) + 1e-12
            dir_term = 1.0 - tf.tanh((yt * yh) / s2)
            p = tf.reduce_mean(g, axis=0)
            cv2 = tf.math.reduce_variance(p) / (tf.square(tf.reduce_mean(p)) + 1e-8)
            return huber(yt, yh) + ALPHA_DIR * tf.reduce_mean(dir_term) + GATE_BALANCE_ALPHA * cv2
    else:
        def loss(yt, yp):
            s2 = tf.math.reduce_variance(yt) + 1e-12
            dir_term = 1.0 - tf.tanh((yt * yp) / s2)
            return huber(yt, yp) + ALPHA_DIR * tf.reduce_mean(dir_term)
    model.compile(optimizer=tf.keras.optimizers.Adam(LR), loss=loss)
    return model

def keras_train_predict(kind, Xw, yw, tr, va, te):
    tf.keras.utils.set_random_seed(int(CFG.get("SEED", 42)))
    Xw = standardize_windows(Xw, tr)
    ep = int(CFG.get("EPOCHS", EPOCHS))
    net = build_keras(kind, (Xw.shape[1], Xw.shape[2]))
    cbs = [tf.keras.callbacks.EarlyStopping(patience=patience(), restore_best_weights=True, min_delta=1e-4)]
    ymu = float(yw[tr].mean()); ysd = float(yw[tr].std()) + 1e-8
    ys = (yw - ymu) / ysd
    net.fit(Xw[tr], ys[tr], validation_data=(Xw[va], ys[va]),
            epochs=ep, batch_size=BATCH, callbacks=cbs, verbose=0)
    return net.predict(Xw[te], verbose=0).squeeze() * ysd + ymu

def keras_fit(kind, Xtr, ytr, epochs=None):
    tf.keras.utils.set_random_seed(int(CFG.get("SEED", 42)))
    net = build_keras(kind, (Xtr.shape[1], Xtr.shape[2]))
    es = tf.keras.callbacks.EarlyStopping(patience=patience(), restore_best_weights=True, min_delta=1e-4)
    ep = int(epochs if epochs is not None else CFG.get("EPOCHS", EPOCHS))
    net.fit(Xtr, ytr, validation_split=0.2, epochs=ep,
            batch_size=BATCH, callbacks=[es], verbose=0)
    return net

def regime_blend(Xw, yw, dates, tr, te, tau=1.0):
    Xw = standardize_windows(Xw, tr)
    tr_idx = np.where(tr)[0]
    phi_tr = np.stack([phi(Xw[i]) for i in tr_idx])
    mu, sd = phi_tr.mean(0), phi_tr.std(0) + 1e-6
    models, profiles, scales = [], [], []
    for s, e in CFG["PERIODS"].values():
        idx = np.where((dates >= s) & (dates <= e) & (dates < CFG["VAL_END"]))[0]
        if len(idx) < 100:
            continue
        ymu = float(yw[idx].mean()); ysd = float(yw[idx].std()) + 1e-8
        models.append(keras_fit("MoE", Xw[idx], (yw[idx] - ymu) / ysd))
        scales.append((ymu, ysd))
        profiles.append((np.mean([phi(Xw[i]) for i in idx], axis=0) - mu) / sd)
    if not models:
        raise RuntimeError("no regime period had >=100 windows for this split")
    prof = np.stack(profiles)
    Xte = Xw[te]
    zphi = np.stack([(phi(Xte[i]) - mu) / sd for i in range(len(Xte))])
    d = np.linalg.norm(zphi[:, None, :] - prof[None, :, :], axis=2)
    alpha = np.exp(-d / tau); alpha /= alpha.sum(1, keepdims=True)
    preds = np.stack([net.predict(Xte, verbose=0).squeeze() * ysd + ymu
                      for net, (ymu, ysd) in zip(models, scales)], axis=1)
    return (alpha * preds).sum(1)

def ensure_regime_pooled(target, horizon, lb):
    key = (target, horizon, lb)
    if key in REGIME_POOLED:
        return REGIME_POOLED[key]
    rng = np.random.default_rng(int(CFG.get("SEED", 42)))
    Xs, ys, ds = [], [], []
    for t in CFG["TICKERS"]:
        f = build_features(t, horizon)
        Xw, yw, dates, _ = make_windows(f, target, lb)
        Xs.append(Xw); ys.append(yw); ds.append(pd.DatetimeIndex(dates))
    X = np.concatenate(Xs); y = np.concatenate(ys)
    D = pd.DatetimeIndex(np.concatenate([d.values for d in ds]))
    tr = D < CFG["TRAIN_END"]
    mu = X[tr].mean(axis=(0, 1), keepdims=True)
    sd = X[tr].std(axis=(0, 1), keepdims=True) + 1e-8
    X = ((X - mu) / sd).astype("float32")
    phi_tr = np.stack([phi(X[i]) for i in np.where(tr)[0][:20000]])
    pmu, psd = phi_tr.mean(0), phi_tr.std(0) + 1e-6
    models, scales, profiles, names = [], [], [], []
    for nm, (s, e) in CFG["PERIODS"].items():
        idx = np.where((D >= s) & (D <= e) & (D < CFG["VAL_END"]))[0]
        if len(idx) < 300:
            continue
        if len(idx) > REGIME_POOL_CAP:
            idx = rng.choice(idx, REGIME_POOL_CAP, replace=False)
        ymu = float(y[idx].mean()); ysd = float(y[idx].std()) + 1e-8
        models.append(keras_fit("MoE", X[idx], (y[idx] - ymu) / ysd, epochs=REGIME_POOL_EPOCHS))
        scales.append((ymu, ysd))
        profiles.append((np.mean([phi(X[i]) for i in idx], axis=0) - pmu) / psd)
        names.append(nm)
    if not models:
        raise RuntimeError("no regime period had enough pooled windows")
    REGIME_POOLED[key] = (models, scales, np.stack(profiles), mu, sd, pmu, psd, names)
    return REGIME_POOLED[key]

def regime_blend_pooled(target, horizon, lb, Xw, te, tau=1.0):
    models, scales, prof, mu, sd, pmu, psd, _names = ensure_regime_pooled(target, horizon, lb)
    Xte = ((Xw[te] - mu) / sd).astype("float32")
    zphi = np.stack([(phi(Xte[i]) - pmu) / psd for i in range(len(Xte))])
    d = np.linalg.norm(zphi[:, None, :] - prof[None, :, :], axis=2)
    alpha = np.exp(-d / tau); alpha /= alpha.sum(1, keepdims=True)
    preds = np.stack([net.predict(Xte, verbose=0).squeeze() * ysd + ymu
                      for net, (ymu, ysd) in zip(models, scales)], axis=1)
    return (alpha * preds).sum(1)

def ensure_regime_pooled_aug(target, horizon, lb):
    key = (target, horizon, lb)
    if key in REGIME_POOLED_AUG:
        return REGIME_POOLED_AUG[key]
    rng = np.random.default_rng(int(CFG.get("SEED", 42)))
    Xs, ys, ds = [], [], []
    for t in CFG["TICKERS"]:
        f = build_features(t, horizon, vol_models=True)
        Xw, yw, dates, _ = make_windows(f, target, lb)
        Xs.append(Xw); ys.append(yw); ds.append(pd.DatetimeIndex(dates))
    X = np.concatenate(Xs); y = np.concatenate(ys)
    D = pd.DatetimeIndex(np.concatenate([d.values for d in ds]))
    tr = D < CFG["TRAIN_END"]
    mu = X[tr].mean(axis=(0, 1), keepdims=True)
    sd = X[tr].std(axis=(0, 1), keepdims=True) + 1e-8
    X = ((X - mu) / sd).astype("float32")
    phi_tr = np.stack([phi(X[i]) for i in np.where(tr)[0][:20000]])
    pmu, psd = phi_tr.mean(0), phi_tr.std(0) + 1e-6
    models, scales, profiles, names = [], [], [], []
    for nm, (s, e) in CFG["PERIODS"].items():
        idx = np.where((D >= s) & (D <= e) & (D < CFG["VAL_END"]))[0]
        if len(idx) < 300:
            continue
        if len(idx) > REGIME_POOL_CAP:
            idx = rng.choice(idx, REGIME_POOL_CAP, replace=False)
        ymu = float(y[idx].mean()); ysd = float(y[idx].std()) + 1e-8
        net = keras_fit("MoE4", X[idx], (y[idx] - ymu) / ysd, epochs=REGIME_POOL_EPOCHS)
        usage = net.predict(X[idx[:512]], verbose=0)[:, 1:].mean(0)
        print(f"[pooled-aug] {nm} ({horizon}): gate usage LSTM/GRU/TR/Lin = "
              f"{np.round(usage, 3).tolist()}" + ("  ⚠ COLLAPSE (>0.8)" if usage.max() > 0.8 else ""))
        models.append(net)
        scales.append((ymu, ysd))
        profiles.append((np.mean([phi(X[i]) for i in idx], axis=0) - pmu) / psd)
        names.append(nm)
    if not models:
        raise RuntimeError("no regime period had enough pooled windows")
    REGIME_POOLED_AUG[key] = (models, scales, np.stack(profiles), mu, sd, pmu, psd, names)
    return REGIME_POOLED_AUG[key]

def regime_blend_pooled_aug(target, horizon, lb, Xw, te, tau=1.0):
    models, scales, prof, mu, sd, pmu, psd, _names = ensure_regime_pooled_aug(target, horizon, lb)
    Xte = ((Xw[te] - mu) / sd).astype("float32")
    zphi = np.stack([(phi(Xte[i]) - pmu) / psd for i in range(len(Xte))])
    d = np.linalg.norm(zphi[:, None, :] - prof[None, :, :], axis=2)
    alpha = np.exp(-d / tau); alpha /= alpha.sum(1, keepdims=True)
    preds = np.stack([net.predict(Xte, verbose=0)[:, 0] * ysd + ymu
                      for net, (ymu, ysd) in zip(models, scales)], axis=1)
    return (alpha * preds).sum(1)

def arima_predict(close, dates, spec, pretest_mask, te_mask):
    close = close.astype(float)
    lr = np.log(close).diff().dropna()
    li, lv = lr.index, lr.values

    if spec["kind"] == "days":
        H = np.full(len(dates), int(spec["n"]), dtype=int)
    else:
        months = int(spec["months"]); ci = close.index
        H = np.empty(len(dates), dtype=int)
        for i, d in enumerate(dates):
            tgt = (pd.Timestamp(d) + pd.DateOffset(months=months)) + pd.offsets.BMonthEnd(0)
            pt = ci.searchsorted(pd.Timestamp(d)); pg = ci.searchsorted(tgt, side="right") - 1
            H[i] = max(1, int(pg - pt))

    pre = np.where(pretest_mask)[0]
    train_end = pd.Timestamp(dates[pre[-1]] if len(pre) else dates[0])
    lr_train = lv[li <= train_end]
    res, p_ar = None, 0
    for od in [(2, 0, 0), (1, 0, 0), (0, 0, 0)]:
        try:
            res = ARIMA(lr_train, order=od).fit(); p_ar = od[0]; break
        except Exception:
            continue
    if res is None:
        const, ar = (float(np.nanmean(lr_train)) if len(lr_train) else 0.0), []
    else:
        pm = dict(zip(res.param_names, res.params))
        const = float(pm.get("const", 0.0))
        ar = [float(pm.get(f"ar.L{i}", 0.0)) for i in range(1, p_ar + 1)]
    p = len(ar)

    out = np.zeros(len(dates), dtype=float)
    for i in np.where(te_mask)[0]:
        pos = li.searchsorted(pd.Timestamp(dates[i]), side="right")
        buf = list(lv[max(0, pos - p):pos]) if p else []
        while len(buf) < p:
            buf.insert(0, const)
        s = 0.0
        for _ in range(int(H[i])):
            nxt = const + sum(ar[j] * buf[-1 - j] for j in range(p)) if p else const
            s += nxt
            if p:
                buf.append(nxt)
        out[i] = s

    return out[te_mask]


In [12]:
def train_model(ticker, model, force=False, target=None, horizon=None):
    target = target or CFG["TARGET"]
    key = (ticker, model, horizon, target)
    if key in PRED_STORE and not force:
        return PRED_STORE[key].copy(), "cached"

    f = build_features(ticker, horizon, vol_models=(model == "Regime_Pooled_Aug"))
    lb = int(CFG["LOOKBACK"])
    Xw, yw, dates, _ = make_windows(f, target, lb)
    dates = pd.DatetimeIndex(dates)
    tr = dates < CFG["TRAIN_END"]
    va = (dates >= CFG["TRAIN_END"]) & (dates < CFG["VAL_END"])
    te = dates >= CFG["VAL_END"]
    if te.sum() == 0:
        raise RuntimeError("no test rows for this split")

    if model == "NAIVE":
        yhat = np.zeros(int(te.sum()), dtype="float32")
    elif model == "ARIMA":
        spec = HORIZONS[horizon]
        yhat = arima_predict(load_price(ticker)["Close"], dates, spec, (tr | va), te)
    elif model == "single_LSTM":
        yhat = keras_train_predict("LSTM", Xw, yw, tr, va, te)
    elif model == "Event_MoE":
        yhat = keras_train_predict("MoE", Xw, yw, tr, va, te)
    elif model == "Event_MoE_Regime":
        yhat = regime_blend(Xw, yw, dates, tr, te)
    elif model == "Regime_Pooled":
        yhat = regime_blend_pooled(target, horizon, lb, Xw, te)
    elif model == "Regime_Pooled_Aug":
        yhat = regime_blend_pooled_aug(target, horizon, lb, Xw, te)
    else:
        raise ValueError(f"model not wired: {model}")

    out = pd.DataFrame({"date": dates[te], "y_true": yw[te], "y_pred": np.asarray(yhat).ravel()})
    PRED_STORE[key] = out
    return out.copy(), "trained"


In [13]:
if not RETRAIN:
    load_saved_predictions()


loaded 840 prediction series from Final predictions/ (40 tickers)


## Test on one ticker

In [14]:
TEST_TICKER, TEST_MODEL, TEST_HORIZON = "AAPL", "Event_MoE", "1ME"
SAMPLE_TICKERS = CFG["TICKERS"][:6]

px = load_price(TEST_TICKER)
print("1) prices:", px.shape, px.index.min().date(), "->", px.index.max().date())

f = build_features(TEST_TICKER, TEST_HORIZON)
print("2) features:", f.shape, "| last 6 cols (macro):", list(f.columns[-6:]))

Xw, yw, _, _ = make_windows(f, CFG["TARGET"], CFG["LOOKBACK"])
print("3) windows:", Xw.shape, "-> (n_windows, lookback, n_features)")

df_pred, how = train_model(TEST_TICKER, TEST_MODEL, horizon=TEST_HORIZON)
print(f"4) forecast [{how}]:", df_pred.shape, "| test dates",
      df_pred['date'].min().date(), "->", df_pred['date'].max().date())
display(df_pred.head())

ic_rows = []
for t in SAMPLE_TICKERS:
    d = train_model(t, TEST_MODEL, horizon=TEST_HORIZON)[0].dropna()
    y, p = d["y_true"].values, d["y_pred"].values
    ic_rows.append(dict(ticker=t, IC=round(float(np.corrcoef(y, p)[0, 1]), 3), n=len(d)))
print("5) per-ticker IC for", TEST_MODEL, TEST_HORIZON)
display(pd.DataFrame(ic_rows))

name, _ = build_and_save(TEST_MODEL, TEST_HORIZON, SAMPLE_TICKERS, K=3, method="TopK")
print("6) portfolio:", name, "| metrics:", {k: round(v, 3) for k, v in
      BOOK_STORE[name]["metrics"].items() if isinstance(v, (int, float))})


1) prices: (6641, 5) 2000-01-03 -> 2026-05-29
2) features: (6561, 14) | last 6 cols (macro): ['is_FOMC', 'days_to_CPI', 'is_CPI', 'days_to_NFP', 'is_NFP', 'y_ret']
3) windows: (6531, 30, 13) -> (n_windows, lookback, n_features)
4) forecast [cached]: (833, 3) | test dates 2023-01-03 -> 2026-04-29


,date,y_true,y_pred
0,2023-01-03,0.165870,0.057160
1,2023-01-04,0.155608,0.050022
2,2023-01-05,0.166270,0.051375
3,2023-01-06,0.130136,0.062102
4,2023-01-09,0.126056,0.058745


5) per-ticker IC for Event_MoE 1ME


,ticker,IC,n
0,NVDA,0.228,833
1,AAPL,0.009,833
2,TSM,0.038,833
3,JPM,0.271,833
4,ASML,0.045,833
5,XOM,-0.219,833


6) portfolio: Event_MoE-1ME-TopK-K3 | metrics: {'ann_ret': 0.592, 'ann_vol': 0.252, 'Sharpe': 2.272, 'Sortino': 3.52, 'Calmar': 2.854, 'max_dd': -0.208, 'hit': 0.547, 'n': 854}


## 5. Train

In [15]:
MODELS = ["NAIVE", "ARIMA", "single_LSTM", "Event_MoE", "Event_MoE_Regime", "Regime_Pooled", "Regime_Pooled_Aug"]
HORIZONS_LIST = ["1D", "1W", "1ME"]

def train_all(models, horizons, tickers):
    for h in horizons:
        for m in models:
            nc = nt = 0
            for t in tickers:
                _, how = train_model(t, m, horizon=h)
                nc += (how == "cached"); nt += (how == "trained")
            print(f"  {m:20s} {h:4s}  trained={nt:2d} cached={nc:2d}")
    print("training done — predictions in PRED_STORE")

if RETRAIN:
    train_all(MODELS, HORIZONS_LIST, TICKERS)
else:
    print("RETRAIN=False — using the", len(PRED_STORE), "saved predictions loaded above")


RETRAIN=False — using the 840 saved predictions loaded above


In [16]:
def write_predictions(tickers, models, horizons):
    for t in tickers:
        with pd.ExcelWriter(os.path.join(PREDS_DIR, f"{t}_preds.xlsx")) as xl:
            wrote = False
            for h in horizons:
                base, extra = None, {}
                for m in models:
                    key = (t, m, h, CFG["TARGET"])
                    if key not in PRED_STORE:
                        continue
                    d = PRED_STORE[key]
                    if base is None:
                        base = d[["date", "y_true"]].copy()
                    extra[f"y_pred_{m}"] = d.set_index("date")["y_pred"]
                if base is None:
                    continue
                merged = base.set_index("date")
                for name, s in extra.items():
                    merged[name] = s
                merged.reset_index().to_excel(xl, sheet_name=h, index=False)
                wrote = True
            if not wrote:
                pd.DataFrame().to_excel(xl, sheet_name="empty", index=False)
    print(f"wrote {len(tickers)} workbooks to {PREDS_DIR}/<TICKER>_preds.xlsx")

if RETRAIN:
    write_predictions(TICKERS, MODELS, HORIZONS_LIST)


## 6. Portfolios

In [ ]:
K_GRID = list(range(2, 21))

def build_books(models, horizons, methods, ks, tickers):
    BOOK_STORE.clear()
    for h in horizons:
        for m in models:
            for meth in methods:
                if meth in ("TopK", "TopKInvVol"):
                    for k in ks:
                        build_and_save(m, h, tickers, K=k, method=meth)
                else:
                    build_and_save(m, h, tickers, method=meth)
    print("books in BOOK_STORE:", len(BOOK_STORE))

build_books(MODELS, HORIZONS_LIST, METHODS, K_GRID, TICKERS)


## 7. Analysis

In [ ]:
def accuracy_table(models, horizons):
    rows = []
    for h in horizons:
        for m in models:
            s = model_pred_summary(m, h)
            rows.append(dict(model=m, horizon=h, n_tickers=s["n_tickers"], IC=s["IC"],
                             rank_IC=s["rank_IC"], dir_acc=s["dir_acc"], RMSE=s["RMSE"],
                             RMSE_vs_naive=s["RMSE_vs_naive"], MAE=s["MAE"], R2=s["R2"],
                             MSE=s["MSE"], bias=s["bias"]))
    return pd.DataFrame(rows)

def per_asset_ic_table(models, horizons):
    rows = []
    for h in horizons:
        for m in models:
            for t in TICKERS:
                try:
                    d = load_predictions(t, m, h).dropna()
                except FileNotFoundError:
                    continue
                y, p = d["y_true"].values.astype(float), d["y_pred"].values.astype(float)
                ic = float(np.corrcoef(y, p)[0, 1]) if (y.std() > 1e-12 and p.std() > 1e-12) else np.nan
                rows.append(dict(ticker=t, model=m, horizon=h, IC=round(ic, 4),
                                 dir_acc=round(float(np.mean(np.sign(y) == np.sign(p))), 4), n=len(d)))
    return pd.DataFrame(rows)

accuracy = accuracy_table(MODELS, HORIZONS_LIST)
print("Mean IC by model × horizon:")
display(accuracy.pivot(index="model", columns="horizon", values="IC").reindex(MODELS)[HORIZONS_LIST].round(4))


In [ ]:
books = sweep_real("Sharpe")
tv = books[(books["method"] == "TopK") & (books["K"] == "7")]
ew = books[books["method"] == "EqualWeight_BuyHold"]
topk_vs_1n = pd.concat([tv, ew]).sort_values(["horizon", "method"]).reset_index(drop=True)

dm_all = pd.concat([dm_predictions_real(h).assign(horizon=h) for h in HORIZONS_LIST],
                   ignore_index=True)
jk_rows = []
for h in HORIZONS_LIST:
    for meth in METHODS:
        j = sharpe_test_portfolio_real(h, meth, 7)
        if len(j):
            jk_rows.append(j.assign(horizon=h, method=meth))
jk_all = pd.concat(jk_rows, ignore_index=True) if jk_rows else pd.DataFrame()

wf_summary = pd.concat([walkforward_stability(MODELS, h, "year")[1].assign(horizon=h)
                        for h in HORIZONS_LIST], ignore_index=True)
wf_by_year = pd.concat([walkforward_stability(MODELS, h, "year")[0].assign(horizon=h)
                        for h in HORIZONS_LIST], ignore_index=True)
per_asset = per_asset_ic_table(MODELS, HORIZONS_LIST)

print("Top books by Sharpe:"); display(books.head(8))
print("Walk-forward (1ME):")
display(wf_summary[wf_summary.horizon == "1ME"][["model","mean_IC","pct_positive","t_stat","verdict"]])


In [ ]:
with pd.ExcelWriter("final_results_df.xlsx") as xl:
    accuracy.to_excel(xl, sheet_name="accuracy", index=False)
    per_asset.to_excel(xl, sheet_name="per_asset_IC", index=False)
    topk_vs_1n.to_excel(xl, sheet_name="portfolio_TopK_vs_1N", index=False)
    books.to_excel(xl, sheet_name="best_books", index=False)
    dm_all.to_excel(xl, sheet_name="significance_DM", index=False)
    jk_all.to_excel(xl, sheet_name="significance_JK", index=False)
    wf_summary.to_excel(xl, sheet_name="walkforward_summary", index=False)
    wf_by_year.to_excel(xl, sheet_name="walkforward_by_year", index=False)
print("wrote final_results_df.xlsx (8 sheets)")


## 8. Figures

In [ ]:
LABEL = {"NAIVE":"NAIVE","ARIMA":"ARIMA","single_LSTM":"single-LSTM","Event_MoE":"Event-MoE",
         "Event_MoE_Regime":"Event-MoE-Regime","Regime_Pooled":"Regime-Pooled","Regime_Pooled_Aug":"Regime-Pooled-Aug"}
FAM = ["ARIMA","single_LSTM","Event_MoE","Event_MoE_Regime","Regime_Pooled","Regime_Pooled_Aug"]
IC = {h: {m: model_pred_summary(m, h)["IC"] for m in ["NAIVE"] + FAM} for h in HORIZONS_LIST}
built = set(BOOK_STORE)
def book_sharpe(name): return BOOK_STORE[name]["metrics"]["Sharpe"] if name in built else None

def fig_ic():
    f = go.Figure()
    for m in FAM: f.add_trace(go.Bar(name=LABEL[m], x=HORIZONS_LIST, y=[IC[h][m] for h in HORIZONS_LIST]))
    f.update_layout(barmode="group", template="plotly_white", height=430,
                    title="Figure 1 — Mean IC by model and horizon", yaxis_title="mean IC",
                    legend=dict(orientation="h", y=-0.2)); return f

def fig_equity(h="1ME"):
    f = go.Figure(); ref = False
    for m, lab in [("Event_MoE","Event-MoE · TopK-7"), ("Regime_Pooled","Regime-Pooled · TopK-7")]:
        n = f"{m}-{h}-TopK-K7"
        if n not in built: continue
        meta = BOOK_STORE[n]; r = meta["returns"]["net_ret"].fillna(0)
        f.add_trace(go.Scatter(x=r.index, y=(10000*(1+r).cumprod()).values, name=lab, mode="lines"))
        if not ref and "bench_1N_rebalanced" in meta["returns"].columns:
            b = meta["returns"]["bench_1N_rebalanced"].fillna(0)
            f.add_trace(go.Scatter(x=b.index, y=(10000*(1+b).cumprod()).values, name="1/N",
                                   line=dict(color="#888", dash="dot"))); ref = True
    f.update_layout(template="plotly_white", height=440, title=f"Figure 2 — Equity curves ({h}, net of cost)",
                    yaxis_title="value (base 10,000)", legend=dict(orientation="h", y=-0.2)); return f

def fig_sharpe_k(h="1ME"):
    ks = sorted({int(re.search(r"-K(\d+)$", b).group(1)) for b in built
                 if b.startswith("Event_MoE-") and f"-{h}-TopK-K" in b and re.search(r"-K(\d+)$", b)})
    f = go.Figure()
    for m, lab in [("Event_MoE","Event-MoE"), ("Regime_Pooled","Regime-Pooled"), ("Regime_Pooled_Aug","Regime-Pooled-Aug")]:
        f.add_trace(go.Scatter(x=ks, y=[book_sharpe(f"{m}-{h}-TopK-K{k}") for k in ks], mode="lines+markers", name=lab))
    s = book_sharpe(f"Event_MoE-{h}-EqualWeight_BuyHold")
    if s: f.add_hline(y=s, line_dash="dot", line_color="#888", annotation_text="1/N")
    f.update_layout(template="plotly_white", height=430, title=f"Figure 3 — Sharpe vs K ({h})",
                    xaxis_title="K (names held)", yaxis_title="Sharpe", legend=dict(orientation="h", y=-0.2)); return f

def fig_best_growth(name="Event_MoE-1ME-TopK-K7"):
    f = go.Figure(); meta = BOOK_STORE[name]; r = meta["returns"]["net_ret"].fillna(0)
    f.add_trace(go.Scatter(x=r.index, y=(10000*(1+r).cumprod()).values, name="Event-MoE · TopK-7",
                           line=dict(color="#2ec27e", width=2)))
    if "bench_1N_rebalanced" in meta["returns"].columns:
        b = meta["returns"]["bench_1N_rebalanced"].fillna(0)
        f.add_trace(go.Scatter(x=b.index, y=(10000*(1+b).cumprod()).values, name="1/N",
                               line=dict(color="#888", dash="dot")))
    f.update_layout(template="plotly_white", height=440, title="Figure 4 — Capital trajectory: best book vs 1/N",
                    yaxis_title="value (base 10,000)", legend=dict(orientation="h", y=-0.2)); return f

def fig_ic_sharpe():
    xs, ys, txt = [], [], []
    for h in HORIZONS_LIST:
        for m in FAM:
            ic = IC[h][m]; s = book_sharpe(f"{m}-{h}-TopK-K7")
            if ic is None or ic != ic or s is None: continue
            xs.append(ic); ys.append(s); txt.append(f"{LABEL[m]} {h}")
    r = float(np.corrcoef(xs, ys)[0, 1]) if len(xs) > 2 else float("nan")
    f = go.Figure(go.Scatter(x=xs, y=ys, mode="markers+text", text=txt, textposition="top center",
                             textfont=dict(size=9), marker=dict(size=10, color="#4c8dff")))
    s1n = book_sharpe("Event_MoE-1ME-EqualWeight_BuyHold")
    if s1n: f.add_hline(y=s1n, line_dash="dot", line_color="#888", annotation_text="1/N")
    f.update_layout(template="plotly_white", height=460,
                    title=f"Figure 5 — IC vs portfolio Sharpe (Pearson r = {r:.2f})",
                    xaxis_title="mean IC", yaxis_title="Sharpe (TopK-7)"); return f, r

def fig_sharpe_heat(h="1ME"):
    meths = ["TopK","ScoreWeighted","TopKInvVol","MaxSharpe","MinVariance","EqualWeight_BuyHold"]
    def sh(m, me): return book_sharpe(f"{m}-{h}-{me}" + ("-K7" if me in ("TopK","TopKInvVol") else ""))
    z = [[sh(m, me) for me in meths] for m in FAM]
    f = go.Figure(go.Heatmap(z=z, x=meths, y=[LABEL[m] for m in FAM], colorscale="RdYlGn",
                             zmid=book_sharpe(f"Event_MoE-{h}-EqualWeight_BuyHold"),
                             text=[[("" if v is None else f"{v:.2f}") for v in row] for row in z],
                             texttemplate="%{text}", colorbar=dict(title="Sharpe")))
    f.update_layout(template="plotly_white", height=380,
                    title=f"Figure 6 — Sharpe by model×method ({h}); green &gt; 1/N", margin=dict(l=130)); return f

def fig_wf_ic(h="1ME"):
    per, _ = walkforward_stability(FAM, h, "year")
    x = [str(w) for w in per["window"]]; f = go.Figure()
    for m in ["Event_MoE","Event_MoE_Regime","Regime_Pooled","Regime_Pooled_Aug","ARIMA","single_LSTM"]:
        if m in per.columns: f.add_trace(go.Bar(name=LABEL[m], x=x, y=per[m].tolist()))
    f.add_hline(y=0, line_color="#888")
    f.update_layout(barmode="group", template="plotly_white", height=420,
                    title=f"Figure 7 — Cross-sectional IC by year ({h})", yaxis_title="IC",
                    legend=dict(orientation="h", y=-0.22)); return f

def fig_ic_heat(h="1ME"):
    ticks = CFG["TICKERS"]
    def ic_tm(m, t):
        try: d = load_predictions(t, m, h).dropna()
        except FileNotFoundError: return None
        y, p = d["y_true"].values.astype(float), d["y_pred"].values.astype(float)
        return float(np.corrcoef(y, p)[0, 1]) if (y.std() > 1e-12 and p.std() > 1e-12) else None
    z = [[ic_tm(m, t) for t in ticks] for m in FAM]
    f = go.Figure(go.Heatmap(z=z, x=ticks, y=[LABEL[m] for m in FAM], colorscale="RdYlGn", zmid=0,
                             colorbar=dict(title="IC")))
    f.update_layout(template="plotly_white", height=320, margin=dict(l=130),
                    title=f"Figure 8 — Per-stock IC across models ({h}); green positive")
    f.update_xaxes(tickangle=90, tickfont=dict(size=8)); return f

def fig_pnl_bar(name="Event_MoE-1ME-TopK-K7"):
    parts = name.split("-"); m, h, meth = parts[0], parts[1], parts[2]
    k = int(parts[3][1:]) if len(parts) > 3 and parts[3].startswith("K") else None
    L = trade_ledger(m, h, CFG["TICKERS"], K=k, method=meth)
    tr, pos = L["trades"], L["positions"]
    real = tr.groupby("ticker")["realized"].sum() if len(tr) else pd.Series(dtype=float)
    fee  = tr.groupby("ticker")["fee"].sum() if len(tr) else pd.Series(dtype=float)
    un = pos.set_index("ticker")["unrealized"] if (pos is not None and len(pos) and "unrealized" in pos.columns) else pd.Series(dtype=float)
    names = sorted(set(real.index) | set(un.index))
    net = {t: float(real.get(t, 0)) + float(un.get(t, 0)) - float(fee.get(t, 0)) for t in names}
    order = sorted(net.items(), key=lambda kv: kv[1])
    sel, seen = [], set()
    for t, v in order[:6] + order[-6:]:
        if t in seen: continue
        seen.add(t); sel.append((t, v))
    sel = sorted(sel, key=lambda kv: kv[1])
    f = go.Figure(go.Bar(x=[v for _, v in sel], y=[t for t, _ in sel], orientation="h",
                         marker_color=["#2ec27e" if v >= 0 else "#ef5b6b" for _, v in sel]))
    f.update_layout(template="plotly_white", height=420, title=f"Figure 9 — Per-name net PnL ({name})",
                    xaxis_title="net PnL ($)"); return f
print("figure builders ready")


In [ ]:
scat_fig, scat_r = fig_ic_sharpe()
FIGS = [("fig1_ic_by_model_horizon", fig_ic()),
        ("fig2_equity_curves_1ME", fig_equity("1ME")),
        ("fig3_sharpe_vs_K_1ME", fig_sharpe_k("1ME")),
        ("fig4_capital_trajectory", fig_best_growth()),
        ("fig5_ic_vs_sharpe_scatter", scat_fig),
        ("fig6_sharpe_heatmap", fig_sharpe_heat("1ME")),
        ("fig7_ic_by_year", fig_wf_ic("1ME")),
        ("fig8_per_stock_ic_heatmap", fig_ic_heat("1ME")),
        ("fig9_per_name_pnl", fig_pnl_bar("Event_MoE-1ME-TopK-K7"))]
for nm, fig in FIGS:
    fig.show()
    try:
        fig.write_image(f"{FIG_DIR}/{nm}.png", scale=2); fig.write_image(f"{FIG_DIR}/{nm}.pdf")
    except Exception as e:
        print(f"[{nm}] static export skipped ({type(e).__name__}); pip install kaleido to save files")
print(f"scatter Pearson r = {scat_r:.3f} ; figures saved to {FIG_DIR}/")
